In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

In [ ]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [ ]:
model=ChatOpenAI(model="gpt-4o-mini")
def chat_node(state:ChatState):
    messages=state["messages"]
    result=model.invoke(messages)
    return {"messages":[result]}

In [ ]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

workflow=graph.compile(checkpointer=checkpointer)
workflow

In [ ]:
thread_id='1'

while True:
    query = input("User: ")
    print("User: ",query)
    if(query.strip().lower() in ['exit','quit','bye']):
        break
    config={"configurable":{"thread_id":thread_id}}
    result = workflow.invoke({"messages": HumanMessage(content=query)}, config=config)
    print("AI: ", result["messages"][-1].content)